In [1]:
#Výpis či je torch nainštalovaný aj pre grafiku a nie iba pre cpu

import torch
from fsspec.config import conf

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU not found")

2.11.0+cu128
True
1
NVIDIA GeForce RTX 5080


In [2]:
# VYTVORENIE RESULTS_WHITE.PNG Z RESULTS.CSV

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

run_dir = Path(r"C:\Users\benko\Desktop\YOLO\runs\detect\detectTP\yolo_tp_depthmap_v1_n")
csv_path = run_dir / "results.csv"

output_dir = Path(r"C:\Users\benko\Desktop\YOLO\runs\detect\validationTP\val_TP_depthmap_v1_n")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "results_white.png"

df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

# x os
x = df["epoch"] if "epoch" in df.columns else range(len(df))

def smooth(y, weight=0.9):
    y = list(y)
    smoothed = []
    last = y[0]
    for point in y:
        last = last * weight + (1 - weight) * point
        smoothed.append(last)
    return smoothed

metrics_to_plot = [
    "train/box_loss",
    "train/cls_loss",
    "train/dfl_loss",
    "metrics/precision(B)",
    "metrics/recall(B)",
    "val/box_loss",
    "val/cls_loss",
    "val/dfl_loss",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)",
]

available = [col for col in metrics_to_plot if col in df.columns]

plt.style.use("default")

fig, axes = plt.subplots(2, 5, figsize=(20, 9))
fig.patch.set_facecolor("white")

for ax, col in zip(axes.flatten(), available):
    y = df[col]

    ax.set_facecolor("white")
    ax.plot(x, y, marker=".", linewidth=1.5, label="results")
    ax.plot(x, smooth(y), linestyle=":", linewidth=2, label="smooth")

    ax.set_title(col)
    ax.grid(True)
    ax.legend()

for ax in axes.flatten()[len(available):]:
    ax.axis("off")

plt.tight_layout()
plt.savefig(output_path, facecolor="white", bbox_inches="tight", dpi=200)
plt.show()

print(f"Results graf uložený ako: {output_path}")

<Figure size 2000x900 with 10 Axes>

Results graf uložený ako: C:\Users\benko\Desktop\YOLO\runs\detect\validationTP\val_TP_depthmap_v1_n\results_white.png


In [1]:
#Trénovanie

from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data="TP26_depthmap_detection/data.yaml",
    epochs=200,
    imgsz=512,
    batch=16,
    device=0,
    patience=30,
    project="detectTP",
    name="yolo_tp_depthmap_v1_n",
    workers=0,
    pretrained=True
)

New https://pypi.org/project/ultralytics/8.4.55 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.31  Python-3.14.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=TP26_depthmap_detection/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name

In [1]:
#Validácia

import matplotlib.pyplot as plt
import matplotlib as mpl
from ultralytics import YOLO

model = YOLO("runs/detect/detectTP/yolo_tp_depthmap_v1_n/weights/best.pt")

plt.style.use("default")
mpl.rcParams["figure.facecolor"] = "white"
mpl.rcParams["axes.facecolor"] = "white"
mpl.rcParams["savefig.facecolor"] = "white"
mpl.rcParams["savefig.edgecolor"] = "white"

metrics = model.val(
    data="TP26_depthmap_detection/data.yaml",
    project="validationTP",
    name="val_TP_depthmap_v1_n",
)

print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

Ultralytics 8.4.31  Python-3.14.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 16303MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 79.610.5 MB/s, size: 9.6 KB)
val: Scanning C:\Users\benko\Desktop\YOLO\TP26_depthmap_detection\valid\labels.cache... 32 images, 17 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 32/32 9.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.5it/s 1.4s4.2s
                   all         32         28      0.867      0.857      0.946      0.554
Speed: 0.7ms preprocess, 1.9ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to C:\Users\benko\Desktop\YOLO\runs\detect\validationTP\val_TP_depthmap_v1_n
mAP50-95: 0.5544370064825259
mAP50: 0.9462623224728488
Precision: 0.8671481579481226
Recall: 0.8571428571428571


In [3]:
#Testovanie

from ultralytics import YOLO

model = YOLO("runs/detect/detectTP/yolo_tp_depthmap_v1_n/weights/best.pt")

results = model.predict(
    source="TP26_depthmap_detection/test/images",
    save=True,
    device=0,
    conf=0.5,
    iou=0.5,
    project="predictTP",
    name="predict_TP_depthmap_v1_n",
    exist_ok=True
)


image 1/15 C:\Users\benko\Desktop\YOLO\TP26_depthmap_detection\test\images\depth_bad_RPi_CM3_11_png.rf.ac991bc8ed54788045f8c90e189cc35a.jpg: 512x512 2 defects, 7.3ms
image 2/15 C:\Users\benko\Desktop\YOLO\TP26_depthmap_detection\test\images\depth_bad_RPi_CM3_18_png.rf.415c6cc7025fc6e95b2d57e01437acd8.jpg: 512x512 1 defect, 6.8ms
image 3/15 C:\Users\benko\Desktop\YOLO\TP26_depthmap_detection\test\images\depth_bad_RPi_CM3_20_png.rf.6f2975da61793d004020745961328ae1.jpg: 512x512 1 defect, 10.7ms
image 4/15 C:\Users\benko\Desktop\YOLO\TP26_depthmap_detection\test\images\depth_bad_RPi_CM3_21_png.rf.d4b13c015ee80906042b6fbc71e6d4a3.jpg: 512x512 1 defect, 8.3ms
image 5/15 C:\Users\benko\Desktop\YOLO\TP26_depthmap_detection\test\images\depth_bad_RPi_CM3_49_png.rf.285a89fd4de792ee85a2d3e8e7b866e1.jpg: 512x512 2 defects, 6.3ms
image 6/15 C:\Users\benko\Desktop\YOLO\TP26_depthmap_detection\test\images\depth_bad_RPi_CM3_4_png.rf.820920cff5e80fa8f1cc8bfe02eae5cf.jpg: 512x512 1 defect, 5.0ms
image 7

In [4]:
#Vykreslovanie metrík

from IPython.display import HTML, display

def show_centered_image(path, width=800):
    display(HTML(f"""
    <div style="text-align: center;">
        <img src="{path}" width="{width}">
    </div>
    """))

show_centered_image(
    "runs/detect/detectTP/yolo_tp_depthmap_v1_n/confusion_matrix.png",
    width=1000,
)

show_centered_image(
    "runs/detect/detectTP/yolo_tp_depthmap_v1_n/confusion_matrix_normalized.png",
    width=1000,
)

show_centered_image(
    "runs/detect/detectTP/yolo_tp_depthmap_v1_n/results.png",
    width=1600,
)

In [5]:
#Trénovanie

from ultralytics import YOLO

model = YOLO("yolo11m.pt")

results = model.train(
    data="TP26_depthmap_detection/data.yaml",
    epochs=200,
    imgsz=512,
    batch=16,
    device=0,
    patience=30,
    project="detectTP",
    name="yolo_tp_depthmap_v1_m",
    workers=0,
    pretrained=True
)

New https://pypi.org/project/ultralytics/8.4.55 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.31  Python-3.14.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=TP26_depthmap_detection/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name

In [1]:
#Validácia

from ultralytics import YOLO

model = YOLO("runs/detect/detectTP/yolo_tp_depthmap_v1_m/weights/best.pt")

metrics = model.val(
    data="TP26_depthmap_detection/data.yaml",
    project="validationTP",
    name="val_TP_depthmap_v1_m",
)

print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

Ultralytics 8.4.31  Python-3.14.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 16303MiB)
YOLO11m summary (fused): 126 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs
val: Fast image access  (ping: 0.10.1 ms, read: 15.62.1 MB/s, size: 9.8 KB)
val: Scanning C:\Users\benko\Desktop\YOLO\TP26_depthmap_detection\valid\labels.cache... 32 images, 17 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 32/32 8.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.7it/s 1.2s3.4s
                   all         32         28      0.989      0.786      0.892      0.583
Speed: 1.0ms preprocess, 4.3ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to C:\Users\benko\Desktop\YOLO\runs\detect\validationTP\val_TP_depthmap_v1_m
mAP50-95: 0.5826339002451341
mAP50: 0.8919808888182637
Precision: 0.9885944665947154
Recall: 0.7857142857142857


In [3]:
#Testovanie

from ultralytics import YOLO

model = YOLO("runs/detect/detectTP/yolo_tp_depthmap_v1_m/weights/best.pt")

results = model.predict(
    source="TP26_depthmap_detection/test/images",
    save=True,
    device=0,
    conf=0.5,
    iou=0.5,
    project="predictTP",
    name="predict_TP_depthmap_v1_m",
    exist_ok=True
)


image 1/15 C:\Users\benko\Desktop\YOLO\TP26_depthmap_detection\test\images\depth_bad_RPi_CM3_11_png.rf.ac991bc8ed54788045f8c90e189cc35a.jpg: 512x512 2 defects, 61.8ms
image 2/15 C:\Users\benko\Desktop\YOLO\TP26_depthmap_detection\test\images\depth_bad_RPi_CM3_18_png.rf.415c6cc7025fc6e95b2d57e01437acd8.jpg: 512x512 1 defect, 7.2ms
image 3/15 C:\Users\benko\Desktop\YOLO\TP26_depthmap_detection\test\images\depth_bad_RPi_CM3_20_png.rf.6f2975da61793d004020745961328ae1.jpg: 512x512 1 defect, 8.4ms
image 4/15 C:\Users\benko\Desktop\YOLO\TP26_depthmap_detection\test\images\depth_bad_RPi_CM3_21_png.rf.d4b13c015ee80906042b6fbc71e6d4a3.jpg: 512x512 1 defect, 6.2ms
image 5/15 C:\Users\benko\Desktop\YOLO\TP26_depthmap_detection\test\images\depth_bad_RPi_CM3_49_png.rf.285a89fd4de792ee85a2d3e8e7b866e1.jpg: 512x512 (no detections), 7.4ms
image 6/15 C:\Users\benko\Desktop\YOLO\TP26_depthmap_detection\test\images\depth_bad_RPi_CM3_4_png.rf.820920cff5e80fa8f1cc8bfe02eae5cf.jpg: 512x512 1 defect, 6.1ms
i

In [5]:
#Vykreslovanie metrík

from IPython.display import HTML, display

def show_centered_image(path, width=800):
    display(HTML(f"""
    <div style="text-align: center;">
        <img src="{path}" width="{width}">
    </div>
    """))

show_centered_image(
    "runs/detect/detectTP/yolo_tp_depthmap_v1_m/confusion_matrix.png",
    width=1000,
)

show_centered_image(
    "runs/detect/detectTP/yolo_tp_depthmap_v1_m/confusion_matrix_normalized.png",
    width=1000,
)

show_centered_image(
    "runs/detect/detectTP/yolo_tp_depthmap_v1_m/results.png",
    width=1600,
)

In [6]:
#Trénovanie

from ultralytics import YOLO

model = YOLO("yolo11x.pt")

results = model.train(
    data="TP26_depthmap_detection/data.yaml",
    epochs=200,
    imgsz=512,
    batch=16,
    device=0,
    patience=30,
    project="detectTP",
    name="yolo_tp_depthmap_v1_x",
    workers=0,
    pretrained=True
)

New https://pypi.org/project/ultralytics/8.4.55 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.31  Python-3.14.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=TP26_depthmap_detection/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11x.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name

In [2]:
#Validácia

from ultralytics import YOLO

model = YOLO("runs/detect/detectTP/yolo_tp_depthmap_v1_x/weights/best.pt")

metrics = model.val(
    data="TP26_depthmap_detection/data.yaml",
    project="validationTP",
    name="val_TP_depthmap_v1_x",
)

print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

Ultralytics 8.4.31  Python-3.14.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 16303MiB)
YOLO11x summary (fused): 191 layers, 56,828,179 parameters, 0 gradients, 194.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 80.58.8 MB/s, size: 9.9 KB)
val: Scanning C:\Users\benko\Desktop\YOLO\TP26_depthmap_detection\valid\labels.cache... 32 images, 17 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 32/32 16.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.1it/s 1.8s5.4s
                   all         32         28      0.914      0.786      0.877       0.51
Speed: 1.0ms preprocess, 7.7ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to C:\Users\benko\Desktop\YOLO\runs\detect\validationTP\val_TP_depthmap_v1_x
mAP50-95: 0.5100516540773905
mAP50: 0.876787074138234
Precision: 0.9141166916230082
Recall: 0.7857142857142857


In [8]:
#Testovanie

from ultralytics import YOLO

model = YOLO("runs/detect/detectTP/yolo_tp_depthmap_v1_x/weights/best.pt")

results = model.predict(
    source="TP26_depthmap_detection/test/images",
    save=True,
    device=0,
    conf=0.5,
    iou=0.5,
    project="predictTP",
    name="predict_TP_depthmap_v1_x",
    exist_ok=True
)


image 1/15 C:\Users\benko\Desktop\YOLO\TP26_depthmap_detection\test\images\depth_bad_RPi_CM3_11_png.rf.ac991bc8ed54788045f8c90e189cc35a.jpg: 512x512 1 defect, 9.1ms
image 2/15 C:\Users\benko\Desktop\YOLO\TP26_depthmap_detection\test\images\depth_bad_RPi_CM3_18_png.rf.415c6cc7025fc6e95b2d57e01437acd8.jpg: 512x512 1 defect, 9.1ms
image 3/15 C:\Users\benko\Desktop\YOLO\TP26_depthmap_detection\test\images\depth_bad_RPi_CM3_20_png.rf.6f2975da61793d004020745961328ae1.jpg: 512x512 1 defect, 11.3ms
image 4/15 C:\Users\benko\Desktop\YOLO\TP26_depthmap_detection\test\images\depth_bad_RPi_CM3_21_png.rf.d4b13c015ee80906042b6fbc71e6d4a3.jpg: 512x512 1 defect, 10.0ms
image 5/15 C:\Users\benko\Desktop\YOLO\TP26_depthmap_detection\test\images\depth_bad_RPi_CM3_49_png.rf.285a89fd4de792ee85a2d3e8e7b866e1.jpg: 512x512 3 defects, 10.0ms
image 6/15 C:\Users\benko\Desktop\YOLO\TP26_depthmap_detection\test\images\depth_bad_RPi_CM3_4_png.rf.820920cff5e80fa8f1cc8bfe02eae5cf.jpg: 512x512 1 defect, 10.5ms
image

In [9]:
#Vykreslovanie metrík

from IPython.display import HTML, display

def show_centered_image(path, width=800):
    display(HTML(f"""
    <div style="text-align: center;">
        <img src="{path}" width="{width}">
    </div>
    """))

show_centered_image(
    "runs/detect/detectTP/yolo_tp_depthmap_v1_x/confusion_matrix.png",
    width=1000,
)

show_centered_image(
    "runs/detect/detectTP/yolo_tp_depthmap_v1_x/confusion_matrix_normalized.png",
    width=1000,
)

show_centered_image(
    "runs/detect/detectTP/yolo_tp_depthmap_v1_x/results.png",
    width=1600,
)